In [1]:
from functions import*



def compute_score(y_true, y_pred):
    return {
        "R2": f"{r2_score(y_true, y_pred):.3f}",
        "MedAE": f"{median_absolute_error(y_true, y_pred):.3f}",
    }

#loading data and setting target and features 
df = pd.read_excel("../data/raw/NIMS_FATIGUE.xlsx")
y = np.log10(df["Nf"])
X = df.drop(columns=["Nf", "DS No. PDF", "Heat", "Testing Material"])
X = pd.get_dummies(X, columns=["Type of test"], drop_first=True)

# We choose to drop DS No.PDF because its an ID and the model could learn from it instead of the physical data 
# We also leave out Heat and Testing Material because we are focusing on the numeric properties 


# setting up train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

#setting up a model with a imputer for missing values and a scalar to standardize the data 
model = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()), 
    ("model", LinearRegression()), 
])
model.fit(X_train, y_train)

#We chose to use Pipeline to make sure that we account for any gaps in the data 
# as well as to prevent data leakage. The mean and std are computed from the training data and then applied to the test data afterwards. 
# This prevents the test information from leaking into the training data. 

y_pred = model.predict(X_test)
print (compute_score(y_test, y_pred))


#comparing different regression models 
models = {
    "Linear regression": LinearRegression(),
    "Random forest": RandomForestRegressor(random_state=42),
}

for name, m in models.items():
    pipe = Pipeline([("impute", SimpleImputer(strategy="median")),
                     ("scale", StandardScaler()),
                     ("model", m)])
    pipe.fit(X_train, y_train)
    print(name,
          "| train:", compute_score(y_train, pipe.predict(X_train)),
          "| test:",  compute_score(y_test,  pipe.predict(X_test)))

{'R2': '0.471', 'MedAE': '0.535'}
Linear regression | train: {'R2': '0.446', 'MedAE': '0.546'} | test: {'R2': '0.471', 'MedAE': '0.535'}
Random forest | train: {'R2': '0.872', 'MedAE': '0.099'} | test: {'R2': '0.692', 'MedAE': '0.184'}
